In [ ]:
# ======================================================================
# CLEAN DUBLIN PLANNING: one ML-ready table, one row per application
# ======================================================================
# Input:  just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv
# Output: just_dublin_dataset\Just_dublin_clean_dataset\planning\planning_dublin_clean.csv
#
# Same cleaning as the national planning notebook, for the 4 Dublin
# councils, keeping the location columns:
#   nation, province, county, local_authority (the council), dublin_district
#
# What this table is for (Student 2):
#   - TF-IDF / n-grams + LDA topics      -> text: DevelopmentDescription
#   - ElasticNet regression              -> target: days_to_decision
#   - Linear SVM classification          -> target: is_refused
#
# Rules: fix the data, do NOT fill missing values / scale / encode here,
# and leave out anything recorded after the decision (leakage).
#
# Needs the Dublin processed notebooks (00 and 02) to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

planning_file = base + r'\just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv'
dublin_clean  = base + r'\just_dublin_dataset\Just_dublin_clean_dataset'

In [ ]:
planning = pd.read_csv(planning_file, low_memory=False)

print(planning.shape)
planning.head(3)

In [ ]:
# Removing extra spaces from all text. Different councils type the same
# value with spaces at the end ('GRANT PERMISSION   '), which made the
# same label look like 3 different ones.

text_cols = []
for col in planning.columns:
    if planning[col].dtype == 'object' or str(planning[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    planning[col] = planning[col].str.strip()
    planning[col] = planning[col].replace('', np.nan)

print('Text columns cleaned:', len(text_cols))

In [ ]:
# Duplicates: some applications appear twice with everything the same except
# OBJECTID (the ArcGIS row number). Those are the same application twice.

cols_without_id = []
for col in planning.columns:
    if col != 'OBJECTID':
        cols_without_id.append(col)

print('Exact duplicates (ignoring OBJECTID):', planning.duplicated(subset=cols_without_id).sum())

rows_before = len(planning)
planning = planning.drop_duplicates(subset=cols_without_id)
print('Removed', rows_before - len(planning), 'rows. Rows left:', len(planning))

In [ ]:
# Dates come back from CSV as text, so converting again.

for col in ['ReceivedDate', 'DecisionDate']:
    planning[col] = pd.to_datetime(planning[col], errors='coerce')

print('Received from', planning['ReceivedDate'].min(), 'to', planning['ReceivedDate'].max())
print('Latest decision date:', planning['DecisionDate'].max())

In [ ]:
# Location check: every application should have a council and a district
# (a real one, or 'Dublin (no district)').

print('Rows without a district:', planning['dublin_district'].isnull().sum())   # should be 0
print('Rows without a council: ', planning['local_authority'].isnull().sum())   # should be 0
print()
print(planning['local_authority'].value_counts())

In [ ]:
# The Decision column is very messy: about 180 different spellings for a
# handful of real outcomes, e.g.
#   'CONDITIONAL', 'GRANT PERMISSION', 'Granted (Conditional)', 'Grant with Conditions'
# Looking at the most common ones first.

planning['Decision'].value_counts(dropna=False).head(40)

In [ ]:
# Grouping all decision labels into a few clear outcomes by looking for key
# words. The order matters (e.g. 'App withdrawn as no AI recd' must be
# Withdrawn, and 'GRANT CERTIFICATE OF EXEMPTION' is not a normal grant).
#
#   Pending    -> no decision yet, or still waiting for further information
#   Invalid    -> application declared invalid
#   Withdrawn  -> applicant withdrew / deemed withdrawn
#   Exempt     -> Section 5 / exemption decisions (not a normal planning decision)
#   Split      -> part granted and part refused
#   Refused
#   Granted    -> conditional or unconditional permission / approval
#   Other      -> anything else (quashed by court, decided by another body...)

def decision_group(text):
    if pd.isnull(text):
        return 'Pending'

    t = text.upper()

    if 'INVA' in t:                     # some labels are cut short: 'APPLICATION DECLARED INVA'
        return 'Invalid'
    if 'WITHDRAW' in t:
        return 'Withdrawn'
    if 'EXEMPT' in t or t.startswith('S5') or 'SECTION 5' in t or 'S179A' in t:
        return 'Exempt'
    if 'SPLIT' in t or ('GRANT' in t and ('REFUSE' in t or 'REF.' in t or 'REF ' in t)):
        return 'Split'
    if ('INFORMATION' in t or 'REQ AI' in t or 'REQUEST AI' in t or 'CLARIFICATION' in t
            or 'EXTENSION OF TIME' in t or 'TIME EXT' in t):
        return 'Pending'
    if 'REFUSE' in t or 'REJECTED' in t:
        return 'Refused'
    if 'GRANT' in t or 'CONDITIONAL' in t or 'APPROV' in t:
        return 'Granted'
    return 'Other'


planning['decision_group'] = planning['Decision'].apply(decision_group)

print(planning['decision_group'].value_counts())

In [ ]:
# Checking the grouping: every raw label next to the group it went into.
# Worth scrolling through to make sure nothing landed in the wrong place.

check = planning.groupby(['decision_group', 'Decision']).size().reset_index(name='count')
check = check.sort_values(['decision_group', 'count'], ascending=[True, False])
check

In [ ]:
# Classification target for the Linear SVM: refused (1) vs granted (0).
# Only applications with a clear Granted or Refused outcome get a label.
# Everything else (pending, invalid, withdrawn, split, exempt) stays empty,
# so it can still be used for the text mining but not for this model.

planning['is_refused'] = np.nan
planning.loc[planning['decision_group'] == 'Granted', 'is_refused'] = 0
planning.loc[planning['decision_group'] == 'Refused', 'is_refused'] = 1

print(planning['is_refused'].value_counts(dropna=False))
print()
print('Refusal rate among decided:', round(planning['is_refused'].mean(), 3))

In [ ]:
# Regression target for ElasticNet: days from received date to decision date.
# Problems found:
#   - some decision dates are BEFORE the received date (negative days)
#   - one decision date is in 2075
# Those can't be real, so they become missing.
# Also only keeping days for real decisions (Granted / Refused / Split); an
# 'invalid' or 'withdrawn' date is not a decision on the application.

planning['days_to_decision'] = (planning['DecisionDate'] - planning['ReceivedDate']).dt.days

print('Negative days:', (planning['days_to_decision'] < 0).sum())
print('Decision dates after Oct 2026:', (planning['DecisionDate'] > '2026-10-31').sum())

planning.loc[planning['days_to_decision'] < 0, 'days_to_decision'] = np.nan
planning.loc[planning['DecisionDate'] > '2026-10-31', 'days_to_decision'] = np.nan
planning.loc[~planning['decision_group'].isin(['Granted', 'Refused', 'Split']), 'days_to_decision'] = np.nan

print(planning['days_to_decision'].describe().round(1))

In [ ]:
# Application type: same problem as Decision (PERMISSION, Permission,
# 'Permission   '...). Grouping into a few clear types.

def application_group(text):
    if pd.isnull(text):
        return np.nan

    t = text.upper()

    if 'CONSEQ' in t or 'ON FOOT' in t or 'FOLLOWING GRANT' in t:
        return 'Consequent on outline'
    if 'EXTENSION' in t:
        return 'Extension of duration'
    if 'EXEMPT' in t or 'SECTION 5' in t or 'SECT. 5' in t or '179' in t:
        return 'Exemption'
    if 'SHD' in t or 'LRD' in t or 'SDZ' in t or 'STRATEGIC' in t:
        return 'Large scale (SHD/LRD/SDZ)'
    if 'PART 8' in t or 'PART VIII' in t or 'PART VLLL' in t or 'PART X' in t or 'PART 10' in t:
        return 'Council own development'
    if 'OUTLINE' in t:
        return 'Outline permission'
    if 'RETENTION' in t and 'PERMISSION AND' in t:
        return 'Permission and retention'
    if 'RETENTION' in t:
        return 'Retention'
    if 'PERMISSION' in t:
        return 'Permission'
    return 'Other'


planning['application_group'] = planning['ApplicationType'].apply(application_group)

print(planning['application_group'].value_counts(dropna=False))

In [ ]:
# One-off house: written as Y, Yes, One, Single House, No... -> 1 / 0.
# Blank stays missing.

print(planning['OneOffHouse'].value_counts(dropna=False))

one_off_map = {'Y': 1, 'Yes': 1, 'One': 1, 'Single House': 1, 'No': 0, 'N': 0}
planning['one_off_house'] = planning['OneOffHouse'].map(one_off_map)

print(planning['one_off_house'].value_counts(dropna=False))

In [ ]:
# Text column: keeping the original description, and adding a lower-case
# version with extra spaces removed (easier for TF-IDF / LDA later).
# Stop words, n-grams etc. are left for the text-mining notebook.

planning['clean_text'] = planning['DevelopmentDescription'].fillna('').str.lower()
planning['clean_text'] = planning['clean_text'].str.split().str.join(' ')

planning['word_count'] = planning['clean_text'].str.split().str.len()

print('Average words:', round(planning['word_count'].mean(), 1))
print('Median words: ', planning['word_count'].median())
print('Empty descriptions:', (planning['word_count'] == 0).sum())

In [ ]:
# Residential flag (the proposal mentions residential-project filters).
# An application counts as residential if it has residential units, is a
# one-off house, or the description mentions a home.

home_words = ['dwelling', 'house', 'apartment', 'residential', 'housing', 'home']

mentions_home = planning['clean_text'].str.contains('|'.join(home_words))

planning['is_residential'] = 0
planning.loc[planning['NumResidentialUnits'] > 0, 'is_residential'] = 1
planning.loc[planning['one_off_house'] == 1, 'is_residential'] = 1
planning.loc[mentions_home, 'is_residential'] = 1

print(planning['is_residential'].value_counts(normalize=True).round(3))

In [ ]:
# Quarter and year from the received date (same format as BER, PPR, rent).

planning['year'] = planning['ReceivedDate'].dt.year
planning['quarter'] = planning['ReceivedDate'].dt.to_period('Q').astype(str)

planning['year'].value_counts().sort_index()

In [ ]:
# Numeric columns: floor area and site area have some huge values
# (site area up to 10 billion). Not deleting them here, just looking.
# Outliers are handled at the modelling stage.

planning[['NumResidentialUnits', 'FloorArea', 'AreaofSite']].describe().round(1)

In [ ]:
# Choosing the columns for the clean table.
# Left out on purpose:
#   GrantDate, ExpiryDate, all Appeal columns, WithdrawnDate -> after the decision (leakage)
#   ApplicationStatus      -> describes where the case ended up (leakage)
#   FIRequestDate, FIRecDate -> only known while the case is being processed
#   DevelopmentAddress, DevelopmentPostcode -> address details; the district is already
#                         taken from them, so not needed (data minimisation)
#   LandUseCode, OneOffKPI, LinkAppDetails, SiteId, ETL_DATE, DecisionDueDate -> mostly empty or admin
#   Decision, ApplicationType, OneOffHouse -> replaced by the cleaned versions

keep = ['OBJECTID', 'nation', 'province', 'county', 'county_id', 'local_authority',
        'dublin_district', 'district_id', 'PlanningAuthority', 'ApplicationNumber',
        'ReceivedDate', 'DecisionDate', 'year', 'quarter',
        'application_group', 'one_off_house', 'NumResidentialUnits', 'FloorArea', 'AreaofSite',
        'is_residential',
        'DevelopmentDescription', 'clean_text', 'word_count',
        'decision_group', 'is_refused', 'days_to_decision']

planning_clean = planning[keep].rename(columns={'OBJECTID': 'application_id'})

print(planning_clean.shape)
planning_clean.head()

In [ ]:
# Missing values left (% per column). Not filling them here.

missing = (planning_clean.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

In [ ]:
planning_clean.info()

In [ ]:
# Saving the clean Dublin planning table.

save_folder = dublin_clean + r'\planning'
os.makedirs(save_folder, exist_ok=True)

planning_clean.to_csv(save_folder + r'\planning_dublin_clean.csv', index=False)
print('Saved', planning_clean.shape, 'to', save_folder + r'\planning_dublin_clean.csv')